# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the FAIR data principles for AI-ready data.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata (accessed directly as object attributes)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}\nVersion: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id` as required.

Let's list the record sets in the dataset and for each, show their fields and example records.

In [ ]:
# List all record sets by @id
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet] if hasattr(dataset.metadata, 'recordSet') else []

if record_set_ids:
    print("Record Sets found:")
    for idx, rs_id in enumerate(record_set_ids):
        print(f"  {idx+1}. {rs_id}")
else:
    print("No record sets found in metadata.\nCheck the Croissant schema or consult dataset documentation.")

In [ ]:
# Show example records and fields for the first record set (if any)
if record_set_ids:
    example_rs = record_set_ids[0]
    print(f"\nFields for record set {example_rs}:")
    fields = []
    for rs in dataset.metadata.recordSet:
        if rs['@id'] == example_rs and 'field' in rs:
            fields = [f['@id'] for f in rs['field']]
            break
    print(fields)
    print("\nExample records:")
    try:
        for rec in dataset.records(record_set=example_rs):
            print(rec)
            break  # show only one record
    except Exception as e:
        print(f"Could not load records: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All record sets and fields are referenced by their `@id`s.

In [ ]:
# Extract data from all record sets
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records from {rs_id}...")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Failed to load {rs_id}: {e}")

# Pick the main record set for further analysis
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"\nSelected for EDA: {main_rs_id}")
    print(f"Fields: {dataframes[main_rs_id].columns.tolist()}")
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps such as filtering and normalizing.
All fields and columns referenced by their `@id`.

In [ ]:
# EDA: Filter, Normalize, and Group
# For demonstration, let's choose 'cr:GAD7_score' as a numeric field, if present
numeric_field_id = 'cr:GAD7_score'
group_field_id = 'cr:gender'  # Use gender as a categorical grouping if present

if main_rs_id and numeric_field_id in dataframes[main_rs_id].columns:
    df = dataframes[main_rs_id]
    threshold = 10
    # Filter records where GAD-7 score > threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize GAD7 score
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by gender if the field exists
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in main record set ({main_rs_id}). Check available columns above.")

## 5. Visualization
Visualize data distributions and relationships between key fields.
All plots reference Columns and Fields by `@id`.

In [ ]:
# Visualize distribution of GAD-7 scores
if main_rs_id and numeric_field_id in dataframes[main_rs_id].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

# Visualize mean GAD-7 score by gender if grouped_df exists
if 'grouped_df' in locals():
    plt.figure(figsize=(6, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel("Gender")
    plt.ylabel("Mean GAD-7 Score")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset is loaded via Croissant schema and parsed with `mlcroissant`.
- We explored record sets, fields (by their `@id`) and visualized GAD-7 scores among survey respondents.
- Example filtering and normalization showcased typical AI-ready data operations.
- Grouping and visualization highlighted potential demographic trends by gender.

Further analysis can explore other scales (PHQ-9, PSQ) and more nuanced correlations using all available fields.